# LeetCode 2970 — Count the Number of Incremovable Subarrays I bằng Python

Link đề: https://leetcode.com/problems/count-the-number-of-incremovable-subarrays-i/ · Độ khó: **Easy** · Chủ đề: Array, Two Pointers, Binary Search, Enumeration

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def incremovableSubarrayCount(self, nums: List[int]) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code (so với vét cạn), nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một **mảng con (subarray)** là **"incremovable"** nếu **xóa nó đi** thì phần còn lại (ghép lại) là dãy **tăng nghiêm ngặt**; đếm số mảng con **không rỗng** như vậy. Mảng rỗng được coi là tăng nghiêm ngặt.
2. Biết **liệt kê mọi mảng con** `nums[l..r]` bằng hai chỉ số, **xóa bằng cắt lát** `nums[:l] + nums[r + 1:]`, và kiểm tra **tăng nghiêm ngặt** bằng `all(...)`.
3. Hiểu cấu trúc phần còn lại: **một tiền tố** `nums[:l]` + **một hậu tố** `nums[r+1:]`; hai phần phải tăng nghiêm ngặt **và chỗ nối** `nums[l-1] < nums[r+1]`.
4. Đi từ cách **vét cạn `O(n³)`** → **đánh dấu tiền tố/hậu tố hợp lệ, kiểm tra `O(1)` mỗi cặp: `O(n²)`** → **hai con trỏ đếm theo tiền tố: `O(n)`** → **hai con trỏ đếm theo hậu tố: `O(n)`**.
5. Tránh các lỗi kinh điển: dùng `<=` thay vì `<` (không nghiêm ngặt), quên chỗ nối giữa tiền tố và hậu tố, quên rằng **xóa cả mảng** (còn lại mảng rỗng) vẫn hợp lệ, và đếm cả mảng con rỗng.

## 2. Đề bài & ví dụ

Cho mảng số nguyên dương `nums` (đánh chỉ số từ `0`). Một mảng con của `nums` được gọi là **incremovable** nếu sau khi **xóa mảng con đó** khỏi `nums`, mảng còn lại trở thành **tăng nghiêm ngặt** (*strictly increasing*, mỗi phần tử **lớn hơn hẳn** phần tử đứng trước).

Hãy trả về **tổng số** mảng con incremovable của `nums`.

**Ghi chú:**

- Mảng con là một dãy phần tử **liên tiếp** và **không rỗng**.
- **Mảng rỗng được coi là tăng nghiêm ngặt.** Vì vậy xóa cả mảng `nums` luôn là một mảng con incremovable.

**Ví dụ 1**

- Input: `nums = [1, 2, 3, 4]`
- Output: `10`
- Giải thích: mảng đã tăng nghiêm ngặt, nên **mọi** mảng con đều incremovable (xóa một đoạn liên tiếp thì phần còn lại vẫn tăng). Số mảng con không rỗng của mảng `4` phần tử là `4 + 3 + 2 + 1 = 10`.

**Ví dụ 2**

- Input: `nums = [6, 5, 7, 8]`
- Output: `7`
- Giải thích: có `7` mảng con incremovable: `[5]`, `[6]`, `[5, 7]`, `[6, 5]`, `[5, 7, 8]`, `[6, 5, 7]`, `[6, 5, 7, 8]`.

**Ví dụ 3**

- Input: `nums = [8, 7, 6, 6]`
- Output: `3`
- Giải thích: chỉ có `[8, 7, 6]` (còn `[6]`), `[7, 6, 6]` (còn `[8]`) và `[8, 7, 6, 6]` (còn rỗng) là incremovable. Chú ý `[6, 6]` ở cuối thì còn `[8, 7]` (giảm) nên không đạt.

**Ràng buộc:** `1 <= nums.length <= 50`; `1 <= nums[i] <= 50`.

In [1]:
from typing import List
import random

examples = [
    ([1, 2, 3, 4], 10),
    ([6, 5, 7, 8], 7),
    ([8, 7, 6, 6], 3),
]
for nums, expected in examples:
    print("nums =", nums, "-> kỳ vọng", expected)

nums = [1, 2, 3, 4] -> kỳ vọng 10
nums = [6, 5, 7, 8] -> kỳ vọng 7
nums = [8, 7, 6, 6] -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1. Mảng con `nums[l..r]` và cách đếm chúng

Mảng con được xác định bởi hai chỉ số `l <= r`: gồm `nums[l], nums[l+1], ..., nums[r]`. Với mảng `n` phần tử có `n(n + 1) / 2` mảng con không rỗng (mỗi cặp `l <= r` một mảng).

In [2]:
nums = [6, 5, 7, 8]
n = len(nums)
pairs = [(l, r) for l in range(n) for r in range(l, n)]
print(len(pairs), n * (n + 1) // 2)                  # 10 10
for l, r in pairs[:4]:
    print((l, r), nums[l:r + 1])                      # nums[l:r+1] là mảng con

10 10
(0, 0) [6]
(0, 1) [6, 5]
(0, 2) [6, 5, 7]
(0, 3) [6, 5, 7, 8]


### 3.2. Xóa một mảng con bằng cắt lát

Xóa `nums[l..r]` nghĩa là giữ **tiền tố** `nums[:l]` và **hậu tố** `nums[r + 1:]`, rồi nối lại: `nums[:l] + nums[r + 1:]`. Nếu `l = 0` thì tiền tố rỗng; nếu `r = n - 1` thì hậu tố rỗng; nếu cả hai thì còn mảng rỗng.

In [3]:
nums = [6, 5, 7, 8]
print(nums[:1] + nums[2:])        # xóa nums[1..1]=[5]  -> [6, 7, 8]
print(nums[:0] + nums[2:])        # xóa nums[0..1]=[6,5] -> [7, 8]
print(nums[:2] + nums[4:])        # xóa nums[2..3]=[7,8] -> [6, 5]
print(nums[:0] + nums[4:])        # xóa cả mảng -> []

[6, 7, 8]
[7, 8]
[6, 5]
[]


### 3.3. Kiểm tra tăng nghiêm ngặt

Mảng `a` tăng nghiêm ngặt nếu `a[i] < a[i + 1]` với mọi `i`. Viết `all(a[i] < a[i + 1] for i in range(len(a) - 1))`. Với mảng **rỗng hoặc một phần tử**, `range(...)` rỗng và `all` của dãy rỗng là **`True`**, đúng ý đề ("mảng rỗng tăng nghiêm ngặt"). Phân biệt: `[1, 2, 2]` **không** tăng nghiêm ngặt (vì `2 < 2` sai), dù nó "không giảm".

In [4]:
def is_strictly_increasing(a):
    return all(a[i] < a[i + 1] for i in range(len(a) - 1))


print(is_strictly_increasing([1, 2, 3]))      # True
print(is_strictly_increasing([1, 2, 2]))      # False: bằng nhau không tính
print(is_strictly_increasing([]), is_strictly_increasing([7]))    # True True
print(all([]))                                 # True: all của dãy rỗng

True
False
True True
True


### 3.4. Phần còn lại = tiền tố + hậu tố: cần cả ba điều kiện

Phần còn lại `nums[:l] + nums[r + 1:]` tăng nghiêm ngặt **khi và chỉ khi**:

1. tiền tố `nums[:l]` tăng nghiêm ngặt;
2. hậu tố `nums[r + 1:]` tăng nghiêm ngặt;
3. **chỗ nối** (nếu cả hai phần không rỗng): `nums[l - 1] < nums[r + 1]`.

Điều kiện 3 dễ bị quên: hai nửa riêng lẻ đều tăng nhưng ghép lại vẫn có thể "gãy" ở chỗ nối.

In [5]:
a, b = [5, 9], [3, 4]
print(is_strictly_increasing(a), is_strictly_increasing(b))    # True True
print(is_strictly_increasing(a + b))                            # False: 9 > 3 ở chỗ nối
print(a[-1] < b[0])                                             # False

True True
False
False


### 3.5. Mảng cờ (flag array) cho tiền tố / hậu tố

Tính một lần: `prefix_ok[l]` = "`nums[:l]` tăng nghiêm ngặt" và `suffix_ok[s]` = "`nums[s:]` tăng nghiêm ngặt" (với `s` từ `0` đến `n`; `nums[n:]` rỗng nên `suffix_ok[n] = True`). Sau đó mỗi câu hỏi về tiền tố/hậu tố chỉ còn tra mảng `O(1)`.

In [6]:
nums = [6, 5, 7, 8]
n = len(nums)
prefix_ok = [True] * (n + 1)
for l in range(2, n + 1):
    prefix_ok[l] = prefix_ok[l - 1] and nums[l - 2] < nums[l - 1]
suffix_ok = [True] * (n + 1)
for s in range(n - 2, -1, -1):
    suffix_ok[s] = suffix_ok[s + 1] and nums[s] < nums[s + 1]
print("prefix_ok:", prefix_ok)       # tiền tố độ dài l: [], [6], [6,5], ...
print("suffix_ok:", suffix_ok)       # hậu tố bắt đầu tại s

prefix_ok: [True, True, False, False, False]
suffix_ok: [False, True, True, True, True]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `nums = [6, 5, 7, 8]`, thử **xóa từng mảng con** và xem phần còn lại:

| xóa `nums[l..r]` | còn lại | tăng nghiêm ngặt? |
|---|---|---|
| `[6]` (`0..0`) | `[5, 7, 8]` | có |
| `[6, 5]` (`0..1`) | `[7, 8]` | có |
| `[6, 5, 7]` (`0..2`) | `[8]` | có |
| `[6, 5, 7, 8]` (`0..3`) | `[]` | có |
| `[5]` (`1..1`) | `[6, 7, 8]` | có |
| `[5, 7]` (`1..2`) | `[6, 8]` | có |
| `[5, 7, 8]` (`1..3`) | `[6]` | có |
| `[7]` (`2..2`) | `[6, 5, 8]` | không (`6 > 5`) |
| `[7, 8]` (`2..3`) | `[6, 5]` | không |
| `[8]` (`3..3`) | `[6, 5, 7]` | không |

Có `7` dòng "có", khớp đáp án `7`. Nhận xét: **chỗ giảm `6 > 5`** buộc mảng con phải **che phủ** `6` hoặc `5` (xóa ít nhất một trong hai). Đó là nguồn gốc của các cách làm nhanh: phần còn lại gồm **tiền tố tăng** và **hậu tố tăng**, và chỉ cần xét **chỗ nối**.

In [7]:
nums = [6, 5, 7, 8]
n = len(nums)
good = []
for l in range(n):
    for r in range(l, n):
        rest = nums[:l] + nums[r + 1:]
        ok = all(rest[i] < rest[i + 1] for i in range(len(rest) - 1))
        print(f"xóa [{l}..{r}] {nums[l:r + 1]!s:14} còn {rest!s:12} -> {'có' if ok else 'không'}")
        if ok:
            good.append((l, r))
print(len(good), "mảng con")

xóa [0..0] [6]            còn [5, 7, 8]    -> có
xóa [0..1] [6, 5]         còn [7, 8]       -> có
xóa [0..2] [6, 5, 7]      còn [8]          -> có
xóa [0..3] [6, 5, 7, 8]   còn []           -> có
xóa [1..1] [5]            còn [6, 7, 8]    -> có
xóa [1..2] [5, 7]         còn [6, 8]       -> có
xóa [1..3] [5, 7, 8]      còn [6]          -> có
xóa [2..2] [7]            còn [6, 5, 8]    -> không
xóa [2..3] [7, 8]         còn [6, 5]       -> không
xóa [3..3] [8]            còn [6, 5, 7]    -> không
7 mảng con


## 5. Cách 1 — Brute Force / baseline: thử mọi mảng con, tạo mảng còn lại, kiểm tra

**Ý tưởng.** Hai vòng lặp `l`, `r` liệt kê mọi mảng con. Với mỗi cặp tạo `rest = nums[:l] + nums[r + 1:]` rồi kiểm tra tăng nghiêm ngặt bằng một vòng. Làm đúng như định nghĩa.

In [8]:
def incremovable_brute(nums: List[int]) -> int:
    n = len(nums)
    count = 0
    for l in range(n):
        for r in range(l, n):                          # xóa nums[l..r]
            rest = nums[:l] + nums[r + 1:]
            if all(rest[i] < rest[i + 1] for i in range(len(rest) - 1)):
                count += 1
    return count


for nums, expected in examples:
    print(nums, "->", incremovable_brute(nums), "(kỳ vọng", expected, ")")

[1, 2, 3, 4] -> 10 (kỳ vọng 10 )
[6, 5, 7, 8] -> 7 (kỳ vọng 7 )
[8, 7, 6, 6] -> 3 (kỳ vọng 3 )


**Độ phức tạp (Cách 1).** Có `O(n²)` cặp `(l, r)`, mỗi cặp tạo mảng còn lại và kiểm tra tốn `O(n)`: **Time `O(n³)`**; mảng `rest`: **Space `O(n)`**. Với `n <= 50` là `~ 10⁵` phép, vẫn chạy nhanh, nhưng nếu `n` lớn (bài II) thì không thể.

**Nút thắt.** Với mỗi cặp ta **tạo lại và quét lại** phần còn lại, trong khi chỉ cần biết "tiền tố có tăng không", "hậu tố có tăng không" và "chỗ nối có tăng không", các câu hỏi này dùng lại được.

## 6. Ý tưởng thuật toán / bước đột phá

Phần còn lại là **tiền tố `nums[:l]`** ghép **hậu tố `nums[r+1:]`**. Nó tăng nghiêm ngặt khi và chỉ khi (mục 3.4):

1. `nums[:l]` tăng nghiêm ngặt: chỉ phụ thuộc `l`, tính trước thành `prefix_ok[l]`;
2. `nums[r+1:]` tăng nghiêm ngặt: chỉ phụ thuộc `r`, tính trước thành `suffix_ok[r+1]`;
3. chỗ nối: `l == 0` hoặc `r + 1 == n` hoặc `nums[l - 1] < nums[r + 1]`.

Mỗi cặp `(l, r)` giờ kiểm tra **`O(1)`**: tổng `O(n²)`. Tiến thêm: `prefix_ok[l]` đúng với `l` từ `0` đến một **giá trị `P` cố định** (tiền tố tăng dài nhất), rồi sai mãi; `suffix_ok[s]` đúng từ một `S` cố định trở đi. Với mỗi `l` hợp lệ, các `s = r + 1` hợp lệ tạo thành **một đoạn liên tục** `[s0, n]` (vì `nums[s]` tăng theo `s` nên điều kiện nối `nums[l-1] < nums[s]` đúng từ một điểm trở đi). Khi `l` tăng, `s0` **không bao giờ lùi**: dùng **hai con trỏ** cho `O(n)`.

## 7. Các cách tiếp theo

### Cách 2 — Mảng cờ tiền tố/hậu tố, kiểm tra `O(1)` mỗi cặp

**Giải thích.** Dựng `prefix_ok` và `suffix_ok` như mục 3.5. Với mỗi `(l, r)` đếm nếu `prefix_ok[l]` và `suffix_ok[r + 1]` và chỗ nối đúng (`l == 0 or r + 1 == n or nums[l - 1] < nums[r + 1]`).

In [9]:
def incremovable_flags(nums: List[int]) -> int:
    n = len(nums)
    prefix_ok = [True] * (n + 1)                       # prefix_ok[l]: nums[:l] tăng nghiêm ngặt
    for l in range(2, n + 1):
        prefix_ok[l] = prefix_ok[l - 1] and nums[l - 2] < nums[l - 1]
    suffix_ok = [True] * (n + 1)                       # suffix_ok[s]: nums[s:] tăng nghiêm ngặt
    for s in range(n - 2, -1, -1):
        suffix_ok[s] = suffix_ok[s + 1] and nums[s] < nums[s + 1]

    count = 0
    for l in range(n):
        for r in range(l, n):
            if not prefix_ok[l] or not suffix_ok[r + 1]:
                continue
            if l == 0 or r + 1 == n or nums[l - 1] < nums[r + 1]:    # chỗ nối
                count += 1
    return count


for nums, expected in examples:
    print(nums, "->", incremovable_flags(nums), "(kỳ vọng", expected, ")")

[1, 2, 3, 4] -> 10 (kỳ vọng 10 )
[6, 5, 7, 8] -> 7 (kỳ vọng 7 )
[8, 7, 6, 6] -> 3 (kỳ vọng 3 )


**Độ phức tạp Cách 2:** **Time `O(n²)`**, **Space `O(n)`**.

### Cách 3 — Hai con trỏ: với mỗi tiền tố giữ lại, đếm các hậu tố tương thích

**Giải thích.**

- `P` = độ dài **tiền tố tăng nghiêm ngặt dài nhất**; `S` = chỉ số **nhỏ nhất** sao cho `nums[S:]` tăng nghiêm ngặt.
- Với mỗi `l` từ `0` đến `P` (giữ lại `nums[:l]`), hậu tố giữ lại bắt đầu tại `s = r + 1`, phải thỏa: `s >= S`, `s >= l + 1` (mảng con xóa `nums[l..s-1]` không rỗng), và nếu `l > 0` và `s < n` thì `nums[s] > nums[l - 1]`. Các `s` thỏa là một đoạn `[s0, n]` nên số cách là `n - s0 + 1`.
- Khi `l` tăng, `s0` chỉ tăng, nên chỉ cần **một con trỏ `s`** chạy một chiều.

In [10]:
def incremovable_two_pointers(nums: List[int]) -> int:
    n = len(nums)
    P = 1                                              # độ dài tiền tố tăng nghiêm ngặt dài nhất
    while P < n and nums[P - 1] < nums[P]:
        P += 1
    S = n - 1                                          # nums[S:] là hậu tố tăng nghiêm ngặt dài nhất
    while S > 0 and nums[S - 1] < nums[S]:
        S -= 1

    ans = 0
    s = S
    for l in range(P + 1):                             # giữ lại nums[:l], l = 0..P
        s = max(s, l + 1)                              # đoạn xóa nums[l..s-1] phải không rỗng
        if l > 0:
            while s < n and nums[s] <= nums[l - 1]:    # chỗ nối: cần nums[s] > nums[l - 1]
                s += 1
        ans += n - s + 1                               # s, s+1, ..., n đều hợp lệ (s = n: hậu tố rỗng)
    return ans


for nums, expected in examples:
    print(nums, "->", incremovable_two_pointers(nums), "(kỳ vọng", expected, ")")

[1, 2, 3, 4] -> 10 (kỳ vọng 10 )
[6, 5, 7, 8] -> 7 (kỳ vọng 7 )
[8, 7, 6, 6] -> 3 (kỳ vọng 3 )


In [11]:
# Quan sát P, S và con trỏ s của ví dụ 2: [6, 5, 7, 8]
nums = [6, 5, 7, 8]
n = len(nums)
P = 1
while P < n and nums[P - 1] < nums[P]:
    P += 1
S = n - 1
while S > 0 and nums[S - 1] < nums[S]:
    S -= 1
print(f"P = {P} (tiền tố {nums[:P]} tăng), S = {S} (hậu tố {nums[S:]} tăng)")
s, total = S, 0
for l in range(P + 1):
    s = max(s, l + 1)
    if l > 0:
        while s < n and nums[s] <= nums[l - 1]:
            s += 1
    total += n - s + 1
    print(f"l={l}: giữ {nums[:l]}, s0={s} -> {n - s + 1} cách (hậu tố bắt đầu từ s0..n)")
print("tổng:", total)

P = 1 (tiền tố [6] tăng), S = 1 (hậu tố [5, 7, 8] tăng)
l=0: giữ [], s0=1 -> 4 cách (hậu tố bắt đầu từ s0..n)
l=1: giữ [6], s0=2 -> 3 cách (hậu tố bắt đầu từ s0..n)
tổng: 7


**Độ phức tạp Cách 3:** `l` chạy tối đa `n + 1` lần, `s` chỉ tăng nên tổng bước của nó `<= n + 1`: **Time `O(n)`**, **Space `O(1)`**.

### Cách 4 — Hai con trỏ theo chiều ngược lại: với mỗi hậu tố giữ lại, đếm các tiền tố tương thích

**Giải thích.** Đối xứng với Cách 3: chọn hậu tố `nums[s:]` giữ lại, `s` chạy từ `n` (hậu tố rỗng) xuống đến `S`. Số tiền tố giữ lại `l` thỏa: `0 <= l <= P`, `l <= s - 1` (đoạn xóa không rỗng), và nếu `l > 0` và `s < n` thì `nums[l - 1] < nums[s]`. Các `l` thỏa là một đoạn `[0, l_max]` nên số cách là `l_max + 1`. Khi `s` **giảm**, `nums[s]` giảm nên `l_max` **chỉ giảm**: một con trỏ thứ hai chạy một chiều.

In [12]:
def incremovable_two_pointers_rev(nums: List[int]) -> int:
    n = len(nums)
    P = 1
    while P < n and nums[P - 1] < nums[P]:
        P += 1
    S = n - 1
    while S > 0 and nums[S - 1] < nums[S]:
        S -= 1

    ans = 0
    l_max = P                                          # l lớn nhất còn hợp lệ
    s = n                                              # hậu tố giữ lại: nums[s:], bắt đầu từ rỗng
    while True:
        # thu nhỏ l_max cho đến khi: l_max <= s - 1 và (l_max == 0 hoặc s == n hoặc nums[l_max-1] < nums[s])
        while l_max > 0 and (l_max > s - 1 or (s < n and nums[l_max - 1] >= nums[s])):
            l_max -= 1
        if s >= 1:                                     # s = 0 nghĩa là không xóa gì: bỏ qua
            ans += l_max + 1
        if s > S:                                      # còn lùi được: nums[s-1:] vẫn tăng nghiêm ngặt
            s -= 1
        else:
            break
    return ans


for nums, expected in examples:
    print(nums, "->", incremovable_two_pointers_rev(nums), "(kỳ vọng", expected, ")")

[1, 2, 3, 4] -> 10 (kỳ vọng 10 )
[6, 5, 7, 8] -> 7 (kỳ vọng 7 )
[8, 7, 6, 6] -> 3 (kỳ vọng 3 )


**Độ phức tạp Cách 4:** `s` đi từ `n` xuống `S`, `l_max` chỉ giảm: **Time `O(n)`**, **Space `O(1)`**.

**Lưu ý khi `nums` đã tăng nghiêm ngặt (như Ví dụ 1):** `P = n`, `S = 0`; cả hai cách vẫn ra `n(n + 1)/2`. Phần test sẽ kiểm tra điều này.

In [13]:
# Code nộp LeetCode cho Cách 2 (O(n^2), đơn giản, hợp ràng buộc n <= 50)
from typing import List

class Solution:
    def incremovableSubarrayCount(self, nums: List[int]) -> int:
        n = len(nums)
        prefix_ok = [True] * (n + 1)
        for l in range(2, n + 1):
            prefix_ok[l] = prefix_ok[l - 1] and nums[l - 2] < nums[l - 1]
        suffix_ok = [True] * (n + 1)
        for s in range(n - 2, -1, -1):
            suffix_ok[s] = suffix_ok[s + 1] and nums[s] < nums[s + 1]
        count = 0
        for l in range(n):
            for r in range(l, n):
                if prefix_ok[l] and suffix_ok[r + 1] and (l == 0 or r + 1 == n or nums[l - 1] < nums[r + 1]):
                    count += 1
        return count

## 8. TRACE từng bước

Ta theo dõi (a) Cách 2 trên `[8, 7, 6, 6]` và (b) Cách 3 (hai con trỏ) trên `[6, 5, 7, 8]`.

In [14]:
def trace_flags(nums):
    n = len(nums)
    prefix_ok = [True] * (n + 1)
    for l in range(2, n + 1):
        prefix_ok[l] = prefix_ok[l - 1] and nums[l - 2] < nums[l - 1]
    suffix_ok = [True] * (n + 1)
    for s in range(n - 2, -1, -1):
        suffix_ok[s] = suffix_ok[s + 1] and nums[s] < nums[s + 1]
    print("nums      =", nums)
    print("prefix_ok =", prefix_ok, " (chỉ số = độ dài tiền tố)")
    print("suffix_ok =", suffix_ok, " (chỉ số = điểm bắt đầu hậu tố)")
    count = 0
    for l in range(n):
        for r in range(l, n):
            pre, suf = prefix_ok[l], suffix_ok[r + 1]
            join = l == 0 or r + 1 == n or nums[l - 1] < nums[r + 1]
            ok = pre and suf and join
            if ok:
                count += 1
            print(f"xóa [{l}..{r}]: tiền tố {pre!s:5} hậu tố {suf!s:5} chỗ nối {join!s:5} -> {'ĐẠT' if ok else '-'} | count={count}")
    print("=> kết quả:", count)
    return count


assert trace_flags([8, 7, 6, 6]) == 3

nums      = [8, 7, 6, 6]
prefix_ok = [True, True, False, False, False]  (chỉ số = độ dài tiền tố)
suffix_ok = [False, False, False, True, True]  (chỉ số = điểm bắt đầu hậu tố)
xóa [0..0]: tiền tố True  hậu tố False chỗ nối True  -> - | count=0
xóa [0..1]: tiền tố True  hậu tố False chỗ nối True  -> - | count=0
xóa [0..2]: tiền tố True  hậu tố True  chỗ nối True  -> ĐẠT | count=1
xóa [0..3]: tiền tố True  hậu tố True  chỗ nối True  -> ĐẠT | count=2
xóa [1..1]: tiền tố True  hậu tố False chỗ nối False -> - | count=2
xóa [1..2]: tiền tố True  hậu tố True  chỗ nối False -> - | count=2
xóa [1..3]: tiền tố True  hậu tố True  chỗ nối True  -> ĐẠT | count=3
xóa [2..2]: tiền tố False hậu tố True  chỗ nối False -> - | count=3
xóa [2..3]: tiền tố False hậu tố True  chỗ nối True  -> - | count=3
xóa [3..3]: tiền tố False hậu tố True  chỗ nối True  -> - | count=3
=> kết quả: 3


In [15]:
def trace_two_pointers(nums):
    n = len(nums)
    P = 1
    while P < n and nums[P - 1] < nums[P]:
        P += 1
    S = n - 1
    while S > 0 and nums[S - 1] < nums[S]:
        S -= 1
    print(f"nums={nums} n={n} P={P} S={S}")
    ans, s = 0, S
    for l in range(P + 1):
        s = max(s, l + 1)
        moved = 0
        if l > 0:
            while s < n and nums[s] <= nums[l - 1]:
                s += 1
                moved += 1
        add = n - s + 1
        ans += add
        print(f"l={l}: giữ {str(nums[:l]):8} s={s} (đã dịch thêm {moved}) -> cộng {add}, ans={ans}")
    print("=> kết quả:", ans)
    return ans


assert trace_two_pointers([6, 5, 7, 8]) == 7
assert trace_two_pointers([8, 7, 6, 6]) == 3

nums=[6, 5, 7, 8] n=4 P=1 S=1
l=0: giữ []       s=1 (đã dịch thêm 0) -> cộng 4, ans=4
l=1: giữ [6]      s=2 (đã dịch thêm 0) -> cộng 3, ans=7
=> kết quả: 7
nums=[8, 7, 6, 6] n=4 P=1 S=3
l=0: giữ []       s=3 (đã dịch thêm 0) -> cộng 2, ans=2
l=1: giữ [8]      s=4 (đã dịch thêm 1) -> cộng 1, ans=3
=> kết quả: 3


Bảng tóm tắt (Cách 3, `[6, 5, 7, 8]`, `n = 4`, `P = 1`, `S = 1`):

| `l` | giữ lại | `s` (sau khi chỉnh) | số cách `n - s + 1` |
|---|---|---|---|
| 0 | `[]` | `max(1, 1) = 1` | `4` (xóa `[0..0]`, `[0..1]`, `[0..2]`, `[0..3]`) |
| 1 | `[6]` | `max(1, 2) = 2`; `nums[2] = 7 > 6` nên giữ `2` | `3` (xóa `[1..1]`, `[1..2]`, `[1..3]`) |
| **tổng** | | | **7** |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (cờ tiền tố/hậu tố, `O(n²)`): đơn giản, đủ nhanh cho `n <= 50`. Đúng class `Solution`, đúng method `incremovableSubarrayCount(self, nums: List[int]) -> int`.

In [16]:
from typing import List

class Solution:
    def incremovableSubarrayCount(self, nums: List[int]) -> int:
        n = len(nums)
        prefix_ok = [True] * (n + 1)                   # prefix_ok[l]: nums[:l] tăng nghiêm ngặt
        for l in range(2, n + 1):
            prefix_ok[l] = prefix_ok[l - 1] and nums[l - 2] < nums[l - 1]
        suffix_ok = [True] * (n + 1)                   # suffix_ok[s]: nums[s:] tăng nghiêm ngặt
        for s in range(n - 2, -1, -1):
            suffix_ok[s] = suffix_ok[s + 1] and nums[s] < nums[s + 1]

        count = 0
        for l in range(n):                             # xóa nums[l..r]
            for r in range(l, n):
                if not prefix_ok[l] or not suffix_ok[r + 1]:
                    continue
                if l == 0 or r + 1 == n or nums[l - 1] < nums[r + 1]:    # chỗ nối
                    count += 1
        return count


sol = Solution()
for nums, expected in examples:
    print(nums, "->", sol.incremovableSubarrayCount(nums))

[1, 2, 3, 4] -> 10
[6, 5, 7, 8] -> 7
[8, 7, 6, 6] -> 3


## 10. Vì sao thuật toán đúng

- **Phân tích phần còn lại.** Xóa `nums[l..r]` (`0 <= l <= r <= n - 1`) để lại `nums[:l]` rồi `nums[r+1:]`. Mảng ghép tăng nghiêm ngặt ⇔ từng nửa tăng nghiêm ngặt **và** (nếu cả hai nửa không rỗng) phần tử cuối của nửa đầu nhỏ hơn phần tử đầu của nửa sau, tức `nums[l-1] < nums[r+1]`. Nếu một nửa rỗng thì không có chỗ nối nào để kiểm tra.
- **Cờ.** `prefix_ok[l]` và `suffix_ok[s]` được tính theo công thức truy hồi: `nums[:l]` tăng ⇔ `nums[:l-1]` tăng và `nums[l-2] < nums[l-1]`; tương tự cho hậu tố. Độ dài `0` và `1` mặc định `True`.
- **Mảng rỗng hợp lệ.** `l = 0, r = n - 1` cho `prefix_ok[0] = True`, `suffix_ok[n] = True`, nối miễn kiểm tra ⇒ luôn được đếm, đúng quy ước "mảng rỗng tăng nghiêm ngặt".
- **Hai con trỏ.** `prefix_ok[l]` đúng ⇔ `l <= P`. `suffix_ok[s]` đúng ⇔ `s >= S` (hoặc `s = n`). Với `l` cố định, điều kiện nối `nums[l-1] < nums[s]` đúng ⇒ đúng với mọi `s' > s` trong hậu tố tăng (vì `nums[s'] > nums[s]`), nên tập `s` hợp lệ là `[s0, n]`; `s0` đơn điệu không giảm theo `l` vì `nums[l-1]` tăng theo `l` (trong tiền tố tăng) và cận `l + 1` tăng.
- **Mảng đã tăng.** `P = n`, mọi cặp đều hợp lệ: `n(n+1)/2` (Ví dụ 1).

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`.

- **Cách 1 (vét cạn):** Time `O(n³)`, Space `O(n)`.
- **Cách 2 (cờ tiền tố/hậu tố):** Time `O(n²)`, Space `O(n)`.
- **Cách 3 (hai con trỏ theo tiền tố):** Time `O(n)`, Space `O(1)`.
- **Cách 4 (hai con trỏ theo hậu tố):** Time `O(n)`, Space `O(1)`.
- Ở đề này `n <= 50`: `O(n³) ~ 1.25·10⁵` còn chạy được, nhưng bài tăng độ khó cùng ý tưởng (bản II, `n` tới `10⁵`) buộc phải dùng Cách 3/4.

## 12. Bộ test

In [17]:
sol = Solution()
impls = {
    "brute": incremovable_brute,
    "flags": incremovable_flags,
    "two_ptr": incremovable_two_pointers,
    "two_ptr_rev": incremovable_two_pointers_rev,
    "submit": sol.incremovableSubarrayCount,
}

for nums, expected in examples:
    for key, fn in impls.items():
        got = fn(list(nums))
        assert got == expected, (key, nums, got, expected)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


### 12.1. Các trường hợp biên

In [18]:
def check_all(nums, expected, label):
    for key, fn in impls.items():
        got = fn(list(nums))
        assert got == expected, (label, key, got, expected)
    print("PASS", label, "->", expected)


check_all([1], 1, "một phần tử: xóa nó, còn rỗng")
check_all([5, 5], 3, "[5,5]: xóa [5] đầu, [5] cuối, hoặc cả hai")
check_all([2, 1], 3, "[2,1]: xóa 2, xóa 1, hoặc cả hai")
check_all([1, 2], 3, "[1,2] đã tăng: mọi mảng con")
check_all([3, 3, 3], 3, "toàn bằng nhau: phần còn lại phải có <= 1 phần tử (xóa >= 2 phần tử)")
check_all([1, 1, 1, 1], 3, "bốn số bằng nhau: xóa 3 (2 cách) hoặc xóa cả 4")
check_all([5, 4, 3, 2, 1], 3, "giảm dần: phần còn lại phải có <= 1 phần tử")
check_all(list(range(1, 51)), 50 * 51 // 2, "n = 50 tăng nghiêm ngặt: n(n+1)/2")
check_all([50] * 50, 3, "n = 50 toàn bằng nhau: xóa 49 (2 cách) hoặc 50")
check_all([1, 3, 2, 4], 8, "một chỗ giảm ở giữa: phải che 3 hoặc 2")
check_all([3, 9, 1, 8], 6, "tiền tố [3], hậu tố [1, 8] đều tăng nhưng chỗ nối 3 > 1 sai khi xóa [9]")

PASS một phần tử: xóa nó, còn rỗng -> 1
PASS [5,5]: xóa [5] đầu, [5] cuối, hoặc cả hai -> 3
PASS [2,1]: xóa 2, xóa 1, hoặc cả hai -> 3
PASS [1,2] đã tăng: mọi mảng con -> 3
PASS toàn bằng nhau: phần còn lại phải có <= 1 phần tử (xóa >= 2 phần tử) -> 3
PASS bốn số bằng nhau: xóa 3 (2 cách) hoặc xóa cả 4 -> 3
PASS giảm dần: phần còn lại phải có <= 1 phần tử -> 3
PASS n = 50 tăng nghiêm ngặt: n(n+1)/2 -> 1275
PASS n = 50 toàn bằng nhau: xóa 49 (2 cách) hoặc 50 -> 3
PASS một chỗ giảm ở giữa: phải che 3 hoặc 2 -> 8
PASS tiền tố [3], hậu tố [1, 8] đều tăng nhưng chỗ nối 3 > 1 sai khi xóa [9] -> 6


### 12.2. Tính chất và phản ví dụ

In [19]:
# Tính chất 1: mảng tăng nghiêm ngặt -> n(n+1)/2
random.seed(5)
for _ in range(200):
    n = random.randint(1, 50)
    nums = sorted(random.sample(range(1, 51), n))
    assert sol.incremovableSubarrayCount(nums) == n * (n + 1) // 2
print("PASS mảng tăng nghiêm ngặt cho n(n+1)/2")

# Tính chất 2: luôn có ít nhất 1 (xóa cả mảng), nhiều nhất n(n+1)/2
for _ in range(500):
    nums = [random.randint(1, 50) for _ in range(random.randint(1, 50))]
    r = sol.incremovableSubarrayCount(nums)
    n = len(nums)
    assert 1 <= r <= n * (n + 1) // 2
print("PASS 1 <= đáp án <= n(n+1)/2")

# Tính chất 3: mỗi mảng con chứa một mảng con incremovable cũng incremovable (đơn điệu theo bao hàm)
def incremovable_pairs(nums):
    n = len(nums)
    res = set()
    for l in range(n):
        for r in range(l, n):
            rest = nums[:l] + nums[r + 1:]
            if all(rest[i] < rest[i + 1] for i in range(len(rest) - 1)):
                res.add((l, r))
    return res

for _ in range(300):
    nums = [random.randint(1, 5) for _ in range(random.randint(1, 10))]
    good = incremovable_pairs(nums)
    for (l, r) in good:
        for l2 in range(0, l + 1):
            for r2 in range(r, len(nums)):
                assert (l2, r2) in good
print("PASS mở rộng mảng con xóa ra rộng hơn vẫn incremovable")

# Tính chất 4: đầu vào không bị sửa
nums = [6, 5, 7, 8]
copy = list(nums)
for key, fn in impls.items():
    fn(nums)
assert nums == copy
print("PASS đầu vào không bị sửa")

# Phản ví dụ 1: dùng <= (không nghiêm ngặt)
def sai_le(nums):
    n, c = len(nums), 0
    for l in range(n):
        for r in range(l, n):
            rest = nums[:l] + nums[r + 1:]
            if all(rest[i] <= rest[i + 1] for i in range(len(rest) - 1)):
                c += 1
    return c

assert sai_le([8, 7, 6, 6]) != 3 and sai_le([3, 3, 3]) != 5
print("PASS phản ví dụ: <= cho", sai_le([8, 7, 6, 6]), "thay vì 3 ở [8, 7, 6, 6]")

# Phản ví dụ 2: quên chỗ nối
def sai_quen_noi(nums):
    n = len(nums)
    prefix_ok = [True] * (n + 1)
    for l in range(2, n + 1):
        prefix_ok[l] = prefix_ok[l - 1] and nums[l - 2] < nums[l - 1]
    suffix_ok = [True] * (n + 1)
    for s in range(n - 2, -1, -1):
        suffix_ok[s] = suffix_ok[s + 1] and nums[s] < nums[s + 1]
    return sum(1 for l in range(n) for r in range(l, n) if prefix_ok[l] and suffix_ok[r + 1])

assert sai_quen_noi([3, 9, 1, 8]) != 6 and sol.incremovableSubarrayCount([3, 9, 1, 8]) == 6
print("PASS phản ví dụ: bỏ điều kiện chỗ nối cho", sai_quen_noi([3, 9, 1, 8]), "thay vì 6 ở [3, 9, 1, 8]")

PASS mảng tăng nghiêm ngặt cho n(n+1)/2
PASS 1 <= đáp án <= n(n+1)/2
PASS mở rộng mảng con xóa ra rộng hơn vẫn incremovable
PASS đầu vào không bị sửa
PASS phản ví dụ: <= cho 4 thay vì 3 ở [8, 7, 6, 6]
PASS phản ví dụ: bỏ điều kiện chỗ nối cho 8 thay vì 6 ở [3, 9, 1, 8]


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [20]:
random.seed(2970)
for trial in range(1500):
    n = random.randint(1, 12)
    hi = random.choice([2, 3, 6, 50])
    nums = [random.randint(1, hi) for _ in range(n)]
    ref = incremovable_brute(list(nums))
    for key, fn in impls.items():
        assert fn(list(nums)) == ref, (key, nums, fn(list(nums)), ref)
print("PASS 1500 bộ ngẫu nhiên nhỏ (n <= 12) - mọi cách khớp vét cạn")

# n tới 50: so với Cách 2 (đã được kiểm chứng ở trên) vì vét cạn O(n^3) chậm hơn
for trial in range(500):
    n = random.randint(13, 50)
    hi = random.choice([3, 10, 50])
    nums = [random.randint(1, hi) for _ in range(n)]
    ref = incremovable_flags(list(nums))
    for key, fn in impls.items():
        if key == "brute":
            continue
        assert fn(list(nums)) == ref, (key, nums)
print("PASS 500 bộ ngẫu nhiên lớn (n tới 50) - các cách khớp nhau")

PASS 1500 bộ ngẫu nhiên nhỏ (n <= 12) - mọi cách khớp vét cạn


PASS 500 bộ ngẫu nhiên lớn (n tới 50) - các cách khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `<=` thay vì `<`

In [21]:
def sai_le_2(nums):
    n, c = len(nums), 0
    for l in range(n):
        for r in range(l, n):
            rest = nums[:l] + nums[r + 1:]
            if all(rest[i] <= rest[i + 1] for i in range(len(rest) - 1)):      # không nghiêm ngặt
                c += 1
    return c


print("sai  :", sai_le_2([8, 7, 6, 6]), "| đúng:", Solution().incremovableSubarrayCount([8, 7, 6, 6]))

sai  : 4 | đúng: 3


**Vì sao sai:** đề yêu cầu **tăng nghiêm ngặt**, mọi phần tử phải **lớn hơn hẳn** phần tử trước. Dùng `<=` coi `[6, 6]` là hợp lệ: ở `[8, 7, 6, 6]`, xóa `[8, 7]` còn `[6, 6]` bị đếm oan (cho `4` thay vì `3`).

### Lỗi 2: Quên chỗ nối giữa tiền tố và hậu tố

In [22]:
def sai_noi(nums):
    n = len(nums)
    pre = [True] * (n + 1)
    for l in range(2, n + 1):
        pre[l] = pre[l - 1] and nums[l - 2] < nums[l - 1]
    suf = [True] * (n + 1)
    for s in range(n - 2, -1, -1):
        suf[s] = suf[s + 1] and nums[s] < nums[s + 1]
    return sum(1 for l in range(n) for r in range(l, n) if pre[l] and suf[r + 1])      # thiếu nums[l-1] < nums[r+1]


print("sai  :", sai_noi([3, 9, 1, 8]), "| đúng:", Solution().incremovableSubarrayCount([3, 9, 1, 8]))

sai  : 8 | đúng: 6


**Vì sao sai:** tiền tố và hậu tố riêng lẻ tăng là **chưa đủ**. Ở `[3, 9, 1, 8]`, xóa `[9]` còn `[3, 1, 8]`: tiền tố `[3]` tăng, hậu tố `[1, 8]` tăng, nhưng ghép lại có `3 > 1` ở **chỗ nối** nên không hợp lệ. Phải kiểm tra thêm `nums[l - 1] < nums[r + 1]`.

### Lỗi 3: Quên rằng xóa cả mảng là hợp lệ

In [23]:
def sai_khong_xoa_het(nums):
    n, c = len(nums), 0
    for l in range(n):
        for r in range(l, n):
            rest = nums[:l] + nums[r + 1:]
            if rest and all(rest[i] < rest[i + 1] for i in range(len(rest) - 1)):   # loại mảng rỗng
                c += 1
    return c


print("sai  :", sai_khong_xoa_het([8, 7, 6, 6]), "| đúng:", Solution().incremovableSubarrayCount([8, 7, 6, 6]))

sai  : 2 | đúng: 3


**Vì sao sai:** đề nói rõ **mảng rỗng là tăng nghiêm ngặt**, nên xóa cả mảng luôn được tính. Điều kiện thừa `if rest and ...` loại mất trường hợp này (đáp án nhỏ hơn đúng một đơn vị ở mọi test).

### Lỗi 4: Đếm cả mảng con rỗng

In [24]:
def sai_dem_rong(nums):
    n, c = len(nums), 0
    for l in range(n + 1):
        for r in range(l - 1, n):                        # r = l - 1: mảng con rỗng
            rest = nums[:l] + nums[r + 1:]
            if all(rest[i] < rest[i + 1] for i in range(len(rest) - 1)):
                c += 1
    return c


print("sai  :", sai_dem_rong([1, 2, 3, 4]), "| đúng:", Solution().incremovableSubarrayCount([1, 2, 3, 4]))

sai  : 15 | đúng: 10


**Vì sao sai:** mảng con phải **không rỗng**, tức `l <= r`. Cho phép `r = l - 1` (xóa "không gì cả") sẽ đếm thêm các trường hợp mà mảng gốc đã tăng nghiêm ngặt. Dùng `for l in range(n): for r in range(l, n)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "xóa một đoạn liên tiếp để phần còn lại thỏa tính chất" → **phần còn lại = tiền tố + hậu tố**; kiểm tra tính chất trên từng phần **và chỗ nối**.
- Mẫu nhớ: **`nums[:l] + nums[r + 1:]`** và điều kiện **`nums[l - 1] < nums[r + 1]`**.
- Tính trước `prefix_ok`, `suffix_ok` để kiểm tra `O(1)`; khi tính chất đơn điệu (tiền tố tăng dài nhất `P`, hậu tố tăng dài nhất `S`) nghĩ ngay tới **hai con trỏ**.
- "Tăng nghiêm ngặt" dùng `<`; "không giảm" dùng `<=`. Mảng rỗng/một phần tử luôn thỏa.
- `all(...)` của dãy rỗng là `True`: đúng ý khi "mảng rỗng được coi là hợp lệ".
- Khi `n` nhỏ (`<= 50`) cứ viết vét cạn đúng trước để làm "đáp án chuẩn" so sánh với cách nhanh.

## 15. Biến thể của bài

1. **Bản II** (`n` tới `10⁵`): bắt buộc dùng hai con trỏ `O(n)`.
2. Đếm số mảng con sao cho xóa đi, phần còn lại **không giảm** (`<=`).
3. Tìm mảng con incremovable **ngắn nhất**.
4. Tìm mảng con incremovable **dài nhất không phải cả mảng** (xóa cả mảng thì tầm thường).
5. Đếm mảng con mà xóa đi thì phần còn lại **giảm nghiêm ngặt**.

Thử biến thể 2, 3 và 5:

In [25]:
def count_non_decreasing_after_remove(nums):
    n, c = len(nums), 0
    for l in range(n):
        for r in range(l, n):
            rest = nums[:l] + nums[r + 1:]
            if all(rest[i] <= rest[i + 1] for i in range(len(rest) - 1)):
                c += 1
    return c


def shortest_incremovable(nums):
    n = len(nums)
    best = None
    for l in range(n):
        for r in range(l, n):
            rest = nums[:l] + nums[r + 1:]
            if all(rest[i] < rest[i + 1] for i in range(len(rest) - 1)):
                if best is None or r - l + 1 < best[0]:
                    best = (r - l + 1, l, r)
    return best


def count_decreasing_after_remove(nums):
    n, c = len(nums), 0
    for l in range(n):
        for r in range(l, n):
            rest = nums[:l] + nums[r + 1:]
            if all(rest[i] > rest[i + 1] for i in range(len(rest) - 1)):
                c += 1
    return c


print("2.", count_non_decreasing_after_remove([8, 7, 6, 6]))        # 4
print("3.", shortest_incremovable([6, 5, 7, 8]))                    # (1, 0, 0): xóa [6]
print("5.", count_decreasing_after_remove([8, 7, 6, 6]))            # 7
assert shortest_incremovable([6, 5, 7, 8]) == (1, 0, 0)
assert shortest_incremovable([1, 2, 3]) == (1, 0, 0)
assert count_non_decreasing_after_remove([8, 7, 6, 6]) == 4         # nhiều hơn đáp án nghiêm ngặt (3)
assert count_decreasing_after_remove([8, 7, 6, 6]) == 7
assert count_decreasing_after_remove([1, 2, 3, 4]) == 3             # chỉ xóa để còn <= 1 phần tử

2. 4
3. (1, 0, 0)
5. 7


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | thử mọi `(l, r)`, tạo `rest`, kiểm tra | `O(n³)` | `O(n)` | hiểu đề, làm "đáp án chuẩn" |
| 2. Cờ tiền tố/hậu tố | `prefix_ok`, `suffix_ok`, chỗ nối `O(1)` | `O(n²)` | `O(n)` | **cách nộp bài** cho `n <= 50` |
| 3. Hai con trỏ theo tiền tố | với mỗi `l`, `s0` chỉ tăng | `O(n)` | `O(1)` | bản II, `n` lớn |
| 4. Hai con trỏ theo hậu tố | với mỗi `s`, `l_max` chỉ giảm | `O(n)` | `O(1)` | đối chiếu Cách 3, đối xứng |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `[1, 2, 3, 4][:1] + [1, 2, 3, 4][3:]`, `all([])`, `is_strictly_increasing([2, 2])`, `Solution().incremovableSubarrayCount([5])`, `Solution().incremovableSubarrayCount([2, 1])`.

**Bài 2 (tính tay).** Với `nums = [1, 4, 2, 3]`: lập bảng `(l, r)` giống mục 4 (xóa gì, còn gì, tăng nghiêm ngặt không) và đếm.

**Bài 3 (điền TODO).** Hoàn thành `incremovable_todo(nums)` theo Cách 1.

In [26]:
def incremovable_todo(nums):
    n = len(nums)
    count = 0
    for l in range(n):
        for r in range(l, n):
            # TODO 1: rest = phần còn lại sau khi xóa nums[l..r]  (dùng nums[:l] + nums[r + 1:])
            # TODO 2: nếu rest tăng nghiêm ngặt thì count += 1
            pass
    return count

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `incremovable_two_pointers2(nums)` theo **Cách 3** (hai con trỏ `P`, `S`, `s`) mà không nhìn code mẫu. Kiểm tra với ba ví dụ trong `examples`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `longest_incremovable_proper(nums)`: độ dài mảng con incremovable **dài nhất nhưng không phải cả mảng** (trả `0` nếu chỉ có cả mảng, tức `n = 1`). (b) Viết `incremovable_list(nums)` trả về **danh sách các cặp `(l, r)`** theo thứ tự `l` rồi `r`. (c) Viết `min_removal_length(nums)`: độ dài **nhỏ nhất** của một mảng con incremovable (luôn tồn tại vì cả mảng luôn đạt).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[1, 4]`, `True`, `False`, `1`, `3`. Bài 3: `rest = nums[:l] + nums[r + 1:]` rồi `if all(rest[i] < rest[i + 1] for i in range(len(rest) - 1)): count += 1`.
- **Mức 2 (vừa):** Bài 2: `[1, 4, 2, 3]`: xóa `[1]` còn `[4, 2, 3]` không; xóa `[1,4]` còn `[2, 3]` có; xóa `[1,4,2]` còn `[3]` có; xóa cả mảng có; xóa `[4]` còn `[1, 2, 3]` có; xóa `[4, 2]` còn `[1, 3]` có; xóa `[4,2,3]` còn `[1]` có; xóa `[2]` còn `[1, 4, 3]` không; xóa `[2,3]` còn `[1, 4]` có; xóa `[3]` còn `[1, 4, 2]` không. Tổng `7`. Bài 4: tính `P`, `S`, rồi vòng `for l in range(P + 1)` với `s = max(s, l + 1)` và `while s < n and nums[s] <= nums[l - 1]`.
- **Mức 3 (sâu):** Bài 5(a): dùng danh sách cặp, lấy `max(r - l + 1)` trong các cặp không phải `(0, n - 1)`. 5(b): thêm `(l, r)` khi đạt. 5(c): mảng con ngắn nhất: tìm `min(r - l + 1)`; cả mảng luôn đạt nên kết quả `<= n`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [27]:
# Bài 1
assert [1, 2, 3, 4][:1] + [1, 2, 3, 4][3:] == [1, 4]
assert all([]) is True and is_strictly_increasing([2, 2]) is False
assert Solution().incremovableSubarrayCount([5]) == 1
assert Solution().incremovableSubarrayCount([2, 1]) == 3

# Bài 2
assert Solution().incremovableSubarrayCount([1, 4, 2, 3]) == 7
assert incremovable_brute([1, 4, 2, 3]) == 7

# Bài 3
def incremovable_todo_ans(nums):
    n = len(nums)
    count = 0
    for l in range(n):
        for r in range(l, n):
            rest = nums[:l] + nums[r + 1:]
            if all(rest[i] < rest[i + 1] for i in range(len(rest) - 1)):
                count += 1
    return count

for nums, expected in examples:
    assert incremovable_todo_ans(nums) == expected

# Bài 4
def incremovable_two_pointers2(nums):
    n = len(nums)
    P = 1
    while P < n and nums[P - 1] < nums[P]:
        P += 1
    S = n - 1
    while S > 0 and nums[S - 1] < nums[S]:
        S -= 1
    ans, s = 0, S
    for l in range(P + 1):
        s = max(s, l + 1)
        if l > 0:
            while s < n and nums[s] <= nums[l - 1]:
                s += 1
        ans += n - s + 1
    return ans

for nums, expected in examples:
    assert incremovable_two_pointers2(nums) == expected

# Bài 5(a), 5(b), 5(c)
def incremovable_list(nums):
    n = len(nums)
    res = []
    for l in range(n):
        for r in range(l, n):
            rest = nums[:l] + nums[r + 1:]
            if all(rest[i] < rest[i + 1] for i in range(len(rest) - 1)):
                res.append((l, r))
    return res

def longest_incremovable_proper(nums):
    n = len(nums)
    lens = [r - l + 1 for (l, r) in incremovable_list(nums) if (l, r) != (0, n - 1)]
    return max(lens) if lens else 0

def min_removal_length(nums):
    return min(r - l + 1 for (l, r) in incremovable_list(nums))

assert incremovable_list([6, 5, 7, 8]) == [(0, 0), (0, 1), (0, 2), (0, 3), (1, 1), (1, 2), (1, 3)]
assert len(incremovable_list([6, 5, 7, 8])) == 7
assert longest_incremovable_proper([6, 5, 7, 8]) == 3
assert longest_incremovable_proper([5]) == 0
assert min_removal_length([6, 5, 7, 8]) == 1
assert min_removal_length([8, 7, 6, 6]) == 3
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (xóa `nums[l..r]`: tiền tố + hậu tố + chỗ nối)

```python
from typing import List

class Solution:
    def incremovableSubarrayCount(self, nums: List[int]) -> int:
        n = len(nums)
        prefix_ok = [True] * (n + 1)
        for l in range(2, n + 1):
            prefix_ok[l] = prefix_ok[l - 1] and nums[l - 2] < nums[l - 1]
        suffix_ok = [True] * (n + 1)
        for s in range(n - 2, -1, -1):
            suffix_ok[s] = suffix_ok[s + 1] and nums[s] < nums[s + 1]
        count = 0
        for l in range(n):
            for r in range(l, n):
                if prefix_ok[l] and suffix_ok[r + 1] and (l == 0 or r + 1 == n or nums[l - 1] < nums[r + 1]):
                    count += 1
        return count
```

**Checklist**

- [ ] Dùng `<` (tăng **nghiêm ngặt**), không phải `<=`?
- [ ] Đã kiểm tra **chỗ nối** `nums[l - 1] < nums[r + 1]` khi cả tiền tố và hậu tố đều không rỗng?
- [ ] Đã tính **xóa cả mảng** (phần còn lại rỗng là hợp lệ)?
- [ ] Mảng con **không rỗng** (`l <= r`), không đếm "xóa không gì cả"?
- [ ] `prefix_ok` / `suffix_ok` có kích thước `n + 1` và `suffix_ok[n] = True` (hậu tố rỗng)?
- [ ] Đã thử: `n = 1`, mảng tăng nghiêm ngặt (`n(n+1)/2`), mảng toàn bằng nhau, mảng giảm dần, `n = 50`?
- [ ] Skeleton đúng `class Solution` / `def incremovableSubarrayCount(self, nums: List[int]) -> int`?